# 🤖 ROTBOTS — Script Generator
## From Sources to Storyboard

**Workshop: "Anatomy of a Content Machine"**  
LI-MA Transformation Digital Art 2026, Amsterdam

---

This notebook takes you through the first half of the ROTBOTS pipeline:

1. **Feed the Machine** — Paste URLs or text, the system scrapes and summarizes them
2. **The Script Writer** — An LLM generates a structured essay with narration and visual directions
3. **Visual Translation** — The essay becomes a storyboard with styles and text-to-video prompts

The output (`prompts.json`) will be used in the next notebook for video generation.

**You don't need to understand the code.** Just fill in your inputs and press ▶️ Play on each cell.

---

## 🔧 Station 0: Setup

Install dependencies and configure the API. Run this cell once at the start.

In [ ]:
# ============================================================
# SETUP — Run this cell first!
# ============================================================

!pip install -q requests beautifulsoup4 pymupdf

import json
import re
import random
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, JSON, HTML

# Create working directory
WORK_DIR = Path("rotbots_output")
WORK_DIR.mkdir(exist_ok=True)

print("✅ Dependencies installed")
print(f"📁 Working directory: {WORK_DIR}")

In [ ]:
# ============================================================
# API KEY — Paste your Groq API key here
# Get one for free at: https://console.groq.com/keys
# ============================================================

GROQ_API_KEY = ""  # <-- Paste your key between the quotes

# LLM Settings
LLM_MODEL = "llama-3.3-70b-versatile"  # Free on Groq
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

# Quick test
if not GROQ_API_KEY:
    print("⚠️  Please paste your Groq API key above!")
    print("   Get one free at: https://console.groq.com/keys")
else:
    print(f"✅ API key configured (model: {LLM_MODEL})")

In [ ]:
# ============================================================
# HELPER FUNCTIONS — The engine under the hood
# You don't need to modify this cell, just run it.
# ============================================================

def query_llm(prompt, system_prompt=None, temperature=None):
    """Send a prompt to the Groq API and return the response."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": temperature or LLM_TEMPERATURE,
        "max_tokens": LLM_MAX_TOKENS
    }
    response = requests.post(
        GROQ_API_URL,
        headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
        json=payload, timeout=120
    )
    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"]
    if "<think>" in content and "</think>" in content:
        content = content.split("</think>")[-1].strip()
    return content

def parse_json_response(response):
    """Extract and parse JSON from LLM response."""
    response = response.strip()
    response = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', response)
    if response.startswith("```"):
        lines = response.split("\n")
        if lines[-1].strip() == "```":
            response = "\n".join(lines[1:-1])
        else:
            response = "\n".join(lines[1:])
    if not response.startswith("[") and not response.startswith("{"):
        json_start = response.find("[")
        obj_start = response.find("{")
        if json_start == -1: json_start = obj_start
        elif obj_start != -1: json_start = min(json_start, obj_start)
        if json_start != -1: response = response[json_start:]
    for end_char in ['}', ']']:
        last_idx = response.rfind(end_char)
        if last_idx != -1:
            truncated = response[:last_idx + 1]
            try: return json.loads(truncated, strict=False)
            except json.JSONDecodeError:
                cleaned = re.sub(r',\s*([}\]])', r'\1', truncated)
                try: return json.loads(cleaned, strict=False)
                except json.JSONDecodeError: pass
    return json.loads(response, strict=False)

def fetch_website_text(url, max_chars=10000):
    """Scrape main text content from a URL."""
    url = url.strip().rstrip('/').strip()
    headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for element in soup(['script', 'style', 'nav', 'header', 'footer', 'aside', 'form']):
        element.decompose()
    main_content = None
    for selector in ['article', 'main', '[role="main"]', '.content', '#content', '.post']:
        main_content = soup.select_one(selector)
        if main_content: break
    if main_content:
        text = main_content.get_text(separator=' ', strip=True)
    else:
        body = soup.find('body')
        text = body.get_text(separator=' ', strip=True) if body else soup.get_text(separator=' ', strip=True)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:max_chars] if len(text) > max_chars else text

def fetch_pdf_text(source, max_chars=10000):
    """Extract text from a PDF (URL or uploaded file path)."""
    import tempfile
    try: import pymupdf as fitz
    except ImportError: import fitz
    pdf_path = None; temp_file = None
    try:
        if source.startswith("http"):
            resp = requests.get(source, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
            resp.raise_for_status()
            temp_file = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
            temp_file.write(resp.content); temp_file.close()
            pdf_path = temp_file.name
        else: pdf_path = source
        doc = fitz.open(pdf_path)
        text_parts = []; total_chars = 0
        for page in doc:
            page_text = page.get_text()
            text_parts.append(page_text); total_chars += len(page_text)
            if total_chars >= max_chars: break
        doc.close()
        text = "\n".join(text_parts)
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text[:max_chars] if len(text) > max_chars else text
    finally:
        if temp_file:
            import os; os.unlink(temp_file.name)

print("✅ Helper functions loaded")

---
## 📥 Station 1: Feed the Machine

Paste your sources below. The machine will scrape websites, read PDFs, and accept raw text.

Then the LLM summarizes each source — compressing pages of content into a few paragraphs.

> **🤔 Think about:** What gets lost in this compression? What does the machine decide is "important"?

In [ ]:
# ============================================================
# YOUR INPUTS — Edit these!
# ============================================================

# What is your video about?
TOPIC = "The history and ethics of AI-generated art"

# Paste your sources here — URLs, PDF links, or raw text
SOURCES = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence_art",
    # "https://example.com/another-article",
    # "path/to/uploaded.pdf",
    # "You can also just paste raw text here as a string.",
]

In [ ]:
# ============================================================
# SCRAPE & SUMMARIZE — Press Play ▶️
# ============================================================

print(f"📥 Processing {len(SOURCES)} source(s) for topic: '{TOPIC}'")
print("=" * 60)
source_texts = []
for i, source in enumerate(SOURCES):
    print(f"\n🔗 Source {i+1}: {source[:80]}..." if len(source) > 80 else f"\n🔗 Source {i+1}: {source}")
    try:
        if source.startswith("http") and source.lower().endswith(".pdf"):
            print("   Type: PDF (URL)"); text = fetch_pdf_text(source)
        elif source.startswith("http"):
            print("   Type: Website"); text = fetch_website_text(source)
        elif source.endswith(".pdf"):
            print("   Type: PDF (local)"); text = fetch_pdf_text(source)
        else:
            print("   Type: Raw text"); text = source
        print(f"   Extracted: {len(text)} characters")
        source_texts.append({"source": source[:100], "text": text})
    except Exception as e:
        print(f"   ❌ Error: {e}")
print(f"\n✅ {len(source_texts)} source(s) loaded successfully")

In [ ]:
# ============================================================
# SUMMARIZE SOURCES — The LLM compresses each source
# ============================================================

print("🧠 Summarizing sources with LLM...")
print("=" * 60)
summaries = []
for i, src in enumerate(source_texts):
    print(f"\n📝 Summarizing source {i+1}/{len(source_texts)}...")
    summary_prompt = f"""Summarize the following text in 3-5 paragraphs. Focus on the key arguments, facts, and insights. This summary will be used as research for a video essay about: "{TOPIC}"

TEXT:
{src['text'][:6000]}

Write a clear, informative summary."""
    summary = query_llm(summary_prompt, system_prompt="You are a research assistant. Summarize accurately and concisely.")
    summaries.append({"source": src["source"], "summary": summary})
    print(f"   ✅ Done ({len(summary)} chars)")
    print(f"   Preview: {summary[:150]}...")

summaries_file = WORK_DIR / "summaries.json"
with open(summaries_file, "w") as f:
    json.dump({"topic": TOPIC, "sources": summaries}, f, indent=2)
print(f"\n✅ Summaries saved to {summaries_file}")
print(f"\n{'=' * 60}")
print("💡 DISCUSSION: Compare these summaries to the original sources.")
print("   What information did the machine keep? What was lost?")

In [ ]:
# ============================================================
# VIEW SUMMARIES
# ============================================================
for i, s in enumerate(summaries):
    display(Markdown(f"### Source {i+1}: `{s['source']}`))
    display(Markdown(s['summary']))
    display(Markdown("---"))

---
## ✍️ Station 2: The Script Writer

Now the LLM writes a **video essay script** based on your topic and the summaries.

It generates:
- A **title** and **thesis** (central argument)
- **Chapters** with sections, each containing:
  - `narration` — what the narrator says
  - `visual_direction` — what the viewer should see
  - `mood` — the emotional tone

> **🤔 Think about:** Who is the "author" here? You chose the topic, but the LLM decided the argument, the structure, and the words.

In [ ]:
# ============================================================
# SCRIPT SETTINGS
# ============================================================
NUM_CHAPTERS = 2
SECTIONS_PER_CHAPTER = 2
print(f"📐 Script structure: {NUM_CHAPTERS} chapters × {SECTIONS_PER_CHAPTER} sections")
print(f"   = {NUM_CHAPTERS * SECTIONS_PER_CHAPTER} content scenes + title card + credits")

In [ ]:
# ============================================================
# GENERATE ESSAY OUTLINE
# ============================================================
print("🧠 Generating essay outline...")
print("=" * 60)
summaries_text = "\n\n".join([f"SOURCE: {s['source']}\n{s['summary']}" for s in summaries])

outline_system = """You are an expert essay writer and researcher. You create compelling, well-structured video essay scripts that weave together insights from multiple sources. Your essays are analytical, engaging, and synthesize insights into original analysis."""

outline_prompt = f"""Create an essay outline for a video essay about: "{TOPIC}"

AVAILABLE RESEARCH:
{summaries_text}

REQUIREMENTS:
- Create exactly {NUM_CHAPTERS} chapters.
- Each chapter should present a distinct argument or perspective
- Build a logical progression from introduction to conclusion
- The essay should feel like a well-researched video essay (think Vox, Kurzgesagt)

Format as JSON:
{{
  "title": "Compelling essay title",
  "thesis": "Central argument in 1-2 sentences",
  "chapters": [
    {{
      "chapter": 1,
      "title": "Chapter Title",
      "summary": "What this chapter argues (2-3 sentences)",
      "key_points": ["point 1", "point 2", "point 3"]
    }}
  ]
}}

Only output the JSON."""

for attempt in range(3):
    try:
        response = query_llm(outline_prompt, outline_system)
        outline = parse_json_response(response)
        break
    except Exception as e:
        print(f"   Attempt {attempt+1}/3 failed: {e}")
        if attempt == 2: raise

if len(outline.get("chapters", [])) > NUM_CHAPTERS:
    outline["chapters"] = outline["chapters"][:NUM_CHAPTERS]

print(f"\n📖 Title: {outline.get('title', 'Untitled')}")
print(f"💡 Thesis: {outline.get('thesis', '')}")
print(f"\n📑 Chapters:")
for ch in outline.get("chapters", []):
    print(f"   {ch['chapter']}: {ch['title']}")

In [ ]:
# ============================================================
# WRITE CHAPTERS
# ============================================================
print("✍️ Writing chapters...")
print("=" * 60)

chapter_system = """You are an expert video essay scriptwriter. You write detailed, engaging narration that synthesizes research and analysis into original commentary.

STYLE RULES:
- Write in a conversational but authoritative tone
- Each section's narration should be 3-6 sentences (enough for ~10-20 seconds of speech)
- Include visual directions that complement the narration
- Create a mood/atmosphere for each section"""

for i, chapter in enumerate(outline["chapters"]):
    print(f"\n📝 Chapter {chapter['chapter']}: {chapter['title']}")
    chapter_prompt = f"""Write the detailed sections for this essay chapter:

ESSAY TOPIC: {TOPIC}
THESIS: {outline.get('thesis', '')}
CHAPTER {chapter['chapter']}: {chapter['title']}
Summary: {chapter.get('summary', '')}
Key points: {json.dumps(chapter.get('key_points', []))}

RESEARCH CONTEXT:
{summaries_text[:4000]}

REQUIREMENTS:
- Write exactly {SECTIONS_PER_CHAPTER} sections.
- Each section needs: narration (3-6 sentences), visual_direction, mood

Format as JSON array:
[{{"section": 1, "narration": "...", "visual_direction": "...", "mood": "..."}}]

Only output the JSON array."""

    response = query_llm(chapter_prompt, chapter_system)
    try:
        sections = parse_json_response(response)
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except Exception as e:
        print(f"   ⚠️ Parse error: {e}")
        sections = [{"section": 1, "narration": chapter['title'], "visual_direction": chapter.get('summary', ''), "mood": "neutral"}]
    if len(sections) > SECTIONS_PER_CHAPTER: sections = sections[:SECTIONS_PER_CHAPTER]
    outline["chapters"][i]["sections"] = sections
    for s in sections:
        print(f"   Section {s.get('section', '?')}: {s.get('narration', '')[:80]}...")

outline["credits"] = {"sources": [s["source"] for s in summaries]}
essay_file = WORK_DIR / "essay_script.json"
with open(essay_file, "w") as f: json.dump(outline, f, indent=2)
total_sections = sum(len(ch.get("sections", [])) for ch in outline["chapters"])
print(f"\n✅ Essay complete! {len(outline['chapters'])} chapters, {total_sections} sections")

In [ ]:
# ============================================================
# VIEW THE ESSAY SCRIPT
# ============================================================
display(Markdown(f"# {outline.get('title', 'Untitled')}"))
display(Markdown(f"*{outline.get('thesis', '')}*"))
display(Markdown("---"))
for chapter in outline["chapters"]:
    display(Markdown(f"## Chapter {chapter['chapter']}: {chapter['title']}"))
    for section in chapter.get("sections", []):
        display(Markdown(f"**Section {section.get('section', '?')}** — *Mood: {section.get('mood', '?')}*"))
        display(Markdown(f"> 🎙️ **Narration:** {section.get('narration', '')}"))
        display(Markdown(f"> 🎬 **Visual:** {section.get('visual_direction', '')}"))
        display(Markdown("---"))

---
## 🎬 Station 3: Visual Translation

Now the essay becomes a **storyboard** — each section becomes a video scene.
Then the **Style Arc** assigns a visual style to each scene.
Finally, the LLM converts each scene into a **text-to-video prompt**.

> **🤔 Think about:** The machine decides what "documentary" or "horror" looks like. What aesthetic assumptions are baked into these style categories?

In [ ]:
# ============================================================
# CHOOSE YOUR STYLE ARC
# ============================================================
STYLE_ARCS = {
    "calm_to_dramatic": {"description": "Starts calm, builds to intense", "early": ["documentary", "nature"], "middle": ["news_report", "sports_commentary"], "late": ["action_movie", "horror"], "credits": ["documentary"]},
    "documentary_journey": {"description": "Classic documentary with increasing energy", "early": ["documentary"], "middle": ["nature", "news_report", "documentary"], "late": ["action_movie", "sports_commentary"], "credits": ["documentary"]},
    "chaos_arc": {"description": "Chaotic brainrot energy, escalating surrealism", "early": ["classic_brainrot", "zoomer_brainrot"], "middle": ["surreal_brainrot", "hyperpop_brainrot"], "late": ["fever_dream_brainrot", "cursed_brainrot"], "credits": ["documentary"]},
}

CONTENT_STYLES = {
    "documentary": {"visual_keywords": "cinematic, professional lighting, steady shots", "audio_keywords": "ambient sounds, subtle music"},
    "nature": {"visual_keywords": "wide nature shots, golden hour, landscapes", "audio_keywords": "nature sounds, bird calls, wind"},
    "news_report": {"visual_keywords": "news studio aesthetic, professional framing", "audio_keywords": "news broadcast audio, serious tone"},
    "sports_commentary": {"visual_keywords": "dynamic angles, action shots, slow motion", "audio_keywords": "crowd cheering, energetic"},
    "action_movie": {"visual_keywords": "dynamic movement, dramatic lighting, wide shots", "audio_keywords": "orchestral hits, impacts"},
    "horror": {"visual_keywords": "dark lighting, shadows, dutch angles, fog", "audio_keywords": "creepy ambience, ominous drones"},
    "classic_brainrot": {"visual_keywords": "chaotic editing, deep fried aesthetic, meme energy", "audio_keywords": "bass boosted, vine booms"},
    "surreal_brainrot": {"visual_keywords": "dreamlike, impossible geometry, AI aesthetic", "audio_keywords": "slowed reverb, distorted"},
    "zoomer_brainrot": {"visual_keywords": "meme aesthetic, ironic, internet culture", "audio_keywords": "TikTok sounds, bass boosted"},
    "hyperpop_brainrot": {"visual_keywords": "maximalist, Y2K, chrome, rainbow gradients", "audio_keywords": "hyperpop beats, synth stabs"},
    "fever_dream_brainrot": {"visual_keywords": "hallucinatory, warped, color bleeding", "audio_keywords": "echoing, time-stretched audio"},
    "cursed_brainrot": {"visual_keywords": "unsettling, low quality, jpeg artifacts", "audio_keywords": "distorted sounds, ominous"},
    "none": {"visual_keywords": "", "audio_keywords": ""}
}

# ⬇️ CHOOSE YOUR ARC HERE ⬇️
CHOSEN_ARC = "calm_to_dramatic"  # Options: calm_to_dramatic, documentary_journey, chaos_arc

arc = STYLE_ARCS[CHOSEN_ARC]
print(f"🎨 Style Arc: {CHOSEN_ARC} — {arc['description']}")

In [ ]:
# ============================================================
# GENERATE STORYBOARD + ASSIGN STYLES
# ============================================================
print("🎬 Generating storyboard...")
scenes = []; scene_num = 1
scenes.append({"scene": scene_num, "scene_type": "title_card", "title": outline.get("title", "Untitled"), "description": outline.get("thesis", "")})
scene_num += 1
for chapter in outline.get("chapters", []):
    ch_num = chapter.get("chapter", 0); ch_title = chapter.get("title", f"Chapter {ch_num}")
    for section in chapter.get("sections", []):
        scenes.append({"scene": scene_num, "scene_type": "content", "title": f"{ch_title} - Part {section.get('section', 1)}", "description": section.get("visual_direction", ""), "narration": section.get("narration", ""), "visual_direction": section.get("visual_direction", ""), "mood": section.get("mood", ""), "chapter": ch_num, "chapter_title": ch_title})
        scene_num += 1
scenes.append({"scene": scene_num, "scene_type": "credits", "title": "Credits", "description": "Sources: " + ", ".join(outline.get("credits", {}).get("sources", []))})

# Assign styles
content_scenes = [s for s in scenes if s["scene_type"] == "content"]
total = len(content_scenes); early_end = max(1, int(total * 0.25)); late_start = max(early_end + 1, int(total * 0.75))
last_style = None
for i, scene in enumerate(content_scenes):
    if i < early_end: pool = arc.get("early", ["documentary"])
    elif i >= late_start: pool = arc.get("late", ["action_movie"])
    else: pool = arc.get("middle", ["news_report"])
    available = [s for s in pool if s != last_style] or pool
    style = random.choice(available)
    scene["assigned_style"] = style
    scene["visual_keywords"] = CONTENT_STYLES.get(style, {}).get("visual_keywords", "")
    scene["audio_keywords"] = CONTENT_STYLES.get(style, {}).get("audio_keywords", "")
    last_style = style

storyboard_file = WORK_DIR / "storyboard.json"
with open(storyboard_file, "w") as f: json.dump(scenes, f, indent=2)
print(f"\n✅ Storyboard: {len(content_scenes)} content scenes + title + credits")
for s in scenes:
    style_tag = f" [{s.get('assigned_style', '')}]" if s.get('assigned_style') else ""
    print(f"   Scene {s['scene']}: [{s['scene_type']}] {s['title']}{style_tag}")

In [ ]:
# ============================================================
# GENERATE T2V PROMPTS
# ============================================================
print("🎥 Converting scenes to text-to-video prompts...")
print("=" * 60)
OPENING_STRUCTURES = [("shot_first", "Start with SHOT TYPE"), ("action_first", "Start with ACTION"), ("setting_first", "Start with ENVIRONMENT"), ("light_first", "Start with LIGHTING"), ("camera_first", "Start with CAMERA MOVEMENT"), ("atmosphere_first", "Start with ATMOSPHERE")]
prompts = []
for scene in scenes:
    if scene["scene_type"] != "content": continue
    scene_num = scene["scene"]; assigned_style = scene.get("assigned_style", "none")
    visual_kw = scene.get("visual_keywords", ""); audio_kw = scene.get("audio_keywords", "")
    rand_opening = random.choice(OPENING_STRUCTURES)
    style_instructions = f"\nApply style: {assigned_style}\nVisual: {visual_kw}\nAudio: {audio_kw}" if assigned_style != "none" else ""
    t2v_system = f"""You are an expert at writing prompts for text-to-video AI models. Write as ONE flowing paragraph of 4-8 sentences covering: shot type, scene setting, action, camera movement, audio/atmosphere.{style_instructions}

RULES: NEVER include text/writing/letters. Include audio descriptions. Focus on visuals and sounds. One or two subjects max."""
    t2v_prompt = f"""Convert to a text-to-video prompt:\n\nVisual Direction: {scene.get('visual_direction', '')}\nMood: {scene.get('mood', '')}\nTitle: {scene.get('title', '')}\n\nStart with: {rand_opening[1]}\n\nOutput ONLY the prompt text."""
    print(f"\n   Scene {scene_num}: {scene.get('title', '')} [{assigned_style}]...")
    response = query_llm(t2v_prompt, t2v_system)
    t2v_text = response.strip().strip('"')
    prompts.append({"scene": scene_num, "title": scene.get("title", ""), "t2v_prompt": t2v_text, "style": assigned_style, "narration": scene.get("narration", ""), "duration": 5})
    print(f"   ✅ {t2v_text[:100]}...")

prompts_file = WORK_DIR / "prompts.json"
with open(prompts_file, "w") as f: json.dump(prompts, f, indent=2)
print(f"\n✅ {len(prompts)} video prompts saved to {prompts_file}")

In [ ]:
# ============================================================
# VIEW FINAL PROMPTS
# ============================================================
display(Markdown("# 🎬 Video Production Plan"))
display(Markdown(f"*{len(prompts)} scenes × ~5 seconds = ~{len(prompts) * 5} seconds of video*"))
display(Markdown("---"))
for p in prompts:
    display(Markdown(f"### Scene {p['scene']}: {p['title']}"))
    display(Markdown(f"**Style:** `{p['style']}` | **Duration:** {p['duration']}s"))
    display(Markdown(f"**🎙️ Narration:** {p['narration']}"))
    display(Markdown(f"**🎥 Video Prompt:** {p['t2v_prompt']}"))
    display(Markdown("---"))
print("\n📁 Output files:"); print("   - summaries.json"); print("   - essay_script.json"); print("   - storyboard.json"); print("   - prompts.json → use in next notebook!")

---
## ⏭️ Next Steps

The `prompts.json` file is ready for the **Video Generation notebook** (`05_Generate.ipynb`).

You can also:
- **Edit the prompts manually** before generating
- **Re-run individual cells** with different settings
- **Use your own footage** instead of AI-generated video

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026*